# CustomRAG: Document Retrieval & Question Answering
An end-to-end Retrieval-Augmented Generation (RAG) pipeline built from scratch. This project ingests custom text documents, chunks and embeds them into a vector space, and uses the Gemini API to answer user queries based strictly on the provided context.

## 1. Environment Setup & Configuration
Importing necessary libraries and initializing the embedding model and Gemini API client. Configuration variables are defined globally for easy tuning.

In [17]:
import time
import json
import os
import numpy as np
import pandas as pd
import gradio as gr
from google import genai
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any

# --- Configuration ---
DOCUMENT_PATH = "null_pointers.txt"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50
DEFAULT_TOP_K = 3
MIN_SIMILARITY_SCORE = 0.0 # Set low for small documents

# --- Environment Agnostic API Key Loading ---
print("Initializing AI clients...")

# 1. Try local environment variable first
api_key = os.environ.get("GEMINI_API_KEY")

# 2. Fallback to Colab secrets if running in Google Colab
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except ImportError:
        pass # Not running in Colab

# 3. Hard fail if key is completely missing
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Please set it in your environment variables or Colab secrets.")

# --- Initialize Clients ---
genai_client = genai.Client(
    api_key=api_key,
    http_options={"timeout": 30000}
)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Clients initialized successfully.")

Initializing AI clients...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Clients initialized successfully.


## 2. Document Ingestion
Loading the custom text document (`null_pointers.txt`). In a production environment, this step would handle parsing PDFs, HTML, or database queries.
## 3. Text Chunking
Splitting the document into overlapping chunks. The overlap is critical to ensure that semantic context isn't lost if a sentence gets split across a boundary.

In [ ]:
def load_document(path: str) -> str:
    """Loads a text document from the local file system."""
    try:
        with open(path, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"Critical Error: Document at {path} not found.")

def create_chunks(text: str, size: int, overlap: int) -> List[Dict[str, Any]]:
    """
    Splits text into overlapping chunks to maintain context at boundaries.
    Overlap ensures that sentences split across chunks are still understood by the LLM.
    """
    chunks = []
    for start_idx in range(0, len(text), size - overlap):
        chunk_text = text[start_idx:start_idx + size]
        chunks.append({
            "id": len(chunks),
            "page": 1,  # Assuming single page for txt, could be dynamic for PDFs
            "text": chunk_text.strip()
        })
    return chunks

# Execute Ingestion
print("Ingesting document...")
raw_text = load_document(DOCUMENT_PATH)
document_chunks = create_chunks(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Successfully created {len(document_chunks)} chunks.")

Ingesting document...
Successfully created 5 chunks.


## 4. Vector Embeddings
Converting text chunks into dense numerical vectors using the `all-MiniLM-L6-v2` model. These vectors capture the semantic meaning of the text.
## 5. Semantic Search (Cosine Similarity)
Implementing the retrieval engine. Because the embeddings are normalized, calculating the dot product is mathematically equivalent to Cosine Similarity. We retrieve the Top-K most relevant chunks for a given query.

In [ ]:
def generate_embeddings(texts: List[str]) -> np.ndarray:
    """Converts text chunks into normalized dense vector embeddings."""
    # Normalization allows us to use dot product for Cosine Similarity later
    return embedding_model.encode(texts, normalize_embeddings=True)

def retrieve_relevant_chunks(query: str, chunks: List[Dict], chunk_vectors: np.ndarray, k: int = 3) -> List[Dict]:
    """
    Retrieves the top-k most relevant chunks using Cosine Similarity.
    Because vectors are normalized, the dot product is mathematically
    equivalent to cosine similarity (higher = more similar).
    """
    query_vector = generate_embeddings([query])[0]
    similarity_scores = chunk_vectors @ query_vector

    # Get indices of the top K highest scores, sorted descending
    top_indices = np.argsort(similarity_scores)[::-1][:k]

    results = []
    for idx in top_indices:
        result = chunks[idx].copy()
        result["score"] = round(float(similarity_scores[idx]), 3)
        results.append(result)

    return results

# Pre-compute embeddings for the document
print("Generating document embeddings...")
chunk_vectors = generate_embeddings([c["text"] for c in document_chunks])
print("Search Engine Ready.")

Generating document embeddings...
Search Engine Ready.


## 6. Prompt Augmentation & Generation
Constructing the final prompt by injecting the retrieved context and sending it to the Gemini API. This ensures the LLM generates answers grounded in our specific document rather than hallucinating.
## 7. Guardrails & Source Citations
Implementing the core RAG pipeline function. We include a similarity threshold guardrail (so the bot refuses to answer if the query is off-topic) and append exact source citations for transparency.

In [ ]:
def query_llm(prompt: str, retries: int = 3) -> str:
    """Sends a prompt to Gemini with exponential backoff for resilience."""
    for attempt in range(retries):
        try:
            response = genai_client.models.generate_content(
                model="gemini-flash-lite-latest",
                contents=prompt,
                config={"max_output_tokens": 300}
            )
            return response.text or ""
        except Exception as e:
            print(f"API Error: {str(e)[:50]}. Retrying ({attempt+1}/{retries})...")
            time.sleep(2 * (attempt + 1))
    return "Error: Could not reach the AI model."

def ask_rag_bot(question: str, k: int = DEFAULT_TOP_K, min_score: float = MIN_SIMILARITY_SCORE) -> str:
    """
    The Main RAG Pipeline:
    1. Retrieve context.
    2. Check similarity threshold.
    3. Augment the prompt.
    4. Generate the answer.
    """
    top_chunks = retrieve_relevant_chunks(question, document_chunks, chunk_vectors, k=k)

    # Guardrail: If the best match is below our threshold, refuse to answer.
    if top_chunks[0]["score"] < min_score:
        return "I apologize, but that information is not in the provided document."

    # Construct the context window
    context = "\n\n".join([f"[{i+1}] {c['text']}" for i, c in enumerate(top_chunks)])

    # System prompt design: Strict instruction to prevent hallucinations
    prompt = (
        "You are an expert assistant. Answer the user's question using ONLY the provided context.\n"
        "If the answer is not contained in the context, say exactly: 'That is not in the document.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    answer = query_llm(prompt)

    # Do not append sources if it refused to answer
    if "not in the document" in answer.lower():
        return answer

    pages_used = sorted({c["page"] for c in top_chunks})
    sources = ", ".join(f"Page {p}" for p in pages_used)

    return f"{answer}\n\n*Sources: {sources}*"

## 8. Web Interface (Gradio)
Wrapping the RAG pipeline into a clean, interactive chat interface using Gradio for the frontend.

In [ ]:
def launch_ui():
    """Boots up the Gradio Chat Interface."""
    interface = gr.ChatInterface(
        fn=ask_rag_bot,
        title="CustomRagBot - Null Pointers Handbook",
        description="Ask me anything about the Null Pointers handbook. I will only answer based on the text.",
        examples=[
            "When and where are the weekly meetings held?",
            "What is the name of the club's mascot?",
            "Who is the current president of the club?"
        ]
    )
    interface.launch(share=True, debug=False)

launch_ui()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff932a5562d8d8e3c7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 9. Evaluation & Tuning
To validate the RAG architecture, we run a test dataset containing "direct" questions, "split" questions (requiring context from multiple chunks), and "absent" questions (which should trigger the refusal guardrail). We compare the RAG accuracy against a baseline plain LLM.

In [ ]:
import json
import pandas as pd

def is_correct(item: Dict[str, Any], reply: str) -> bool:
    """
    Evaluates if the LLM response matches the expected keywords or refusal behavior.
    """
    r = reply.lower()

    # If the question is absent from the document, the bot MUST refuse to answer.
    if item["type"] == "absent":
        refusal_phrases = ["not in the provided document", "not in the document", "don't know"]
        return any(phrase in r for phrase in refusal_phrases)

    # Otherwise, verify all expected keywords are present in the response.
    return all(keyword.lower() in r for keyword in item["keywords"])

def run_evaluation(test_set: List[Dict[str, Any]], k: int = DEFAULT_TOP_K) -> pd.DataFrame:
    """
    Compares a baseline LLM against the RAG pipeline across the test dataset.
    """
    results = []

    print(f"Running evaluation on {len(test_set)} questions...\n")
    for item in test_set:
        query = item["question"]

        # 1. Baseline: Ask the LLM without any context
        baseline_answer = query_llm(query)

        # 2. RAG: Ask our pipeline with retrieved context
        rag_answer = ask_rag_bot(query, k=k, min_score=MIN_SIMILARITY_SCORE)

        results.append({
            "question": query,
            "type": item["type"],
            "plain_correct": is_correct(item, baseline_answer),
            "rag_correct": is_correct(item, rag_answer)
        })

        # Sleep briefly to respect API rate limits
        time.sleep(2)

    df_results = pd.DataFrame(results)

    print("--- Final Accuracy ---")
    print(f"Plain LLM (No Context) : {df_results.plain_correct.mean() * 100:.1f}%")
    print(f"RAG Bot (With Context) : {df_results.rag_correct.mean() * 100:.1f}%\n")

    return df_results

# --- Execute Evaluation ---
# Ensure test_set.json is uploaded to the Colab environment
test_set = json.load(open("test_set.json"))
df_results = run_evaluation(test_set)

# Display the final results table
df_results

Running evaluation on 8 questions...

--- Final Accuracy ---
Plain LLM (No Context) : 0.0%
RAG Bot (With Context) : 87.5%



,question,type,plain_correct,rag_correct
0,When and where are the weekly meetings held?,direct,False,True
1,What is the name of the club's mascot?,direct,False,True
2,What is the official snack of the club and wha...,direct,False,True
3,What is the secret password to access the hidd...,direct,False,True
4,What is the penalty for being late to a meetin...,split,False,True
5,"Who established the club in 2018, and what pro...",split,False,False
6,Who is the current president of the Null Point...,absent,False,True
7,How much is the annual membership fee for the ...,absent,False,True
